In [7]:
import random
import numpy as np

class Config:
    MAZE_SIZE = 15          # Road network ke liye 15 ya 20 size zyada achha lagta hai
    STREET_SPACING = 3      # Har 3 blocks baad ek sarak (road) banegi

class RoadMazeGenerator:
    """
    Ek realistic city grid map generator jo proper sarakein (roads),
    intersections (chowk), aur blocks banata hai.
    """

    @staticmethod
    def count_distinct_paths(maze, start, goal, max_paths_to_find=5):
        """ BFS/DFS check karne ke liye ki Start se Goal tak rasta hai ya nahi """
        size = maze.shape[0]
        paths_found = 0
        visited = np.zeros_like(maze, dtype=bool)

        def dfs(curr):
            nonlocal paths_found
            if paths_found >= max_paths_to_find:
                return
            if curr == goal:
                paths_found += 1
                return

            r, c = curr
            for r_off, c_off in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                nr, nc = r + r_off, c + c_off
                if 0 <= nr < size and 0 <= nc < size:
                    if maze[nr, nc] == 0 and not visited[nr, nc]:
                        visited[nr, nc] = True
                        dfs((nr, nc))
                        visited[nr, nc] = False

        visited[start[0], start[1]] = True
        dfs(start)
        return paths_found

    @classmethod
    def generate_road_network(cls, size, spacing):
        attempts = 0
        while True:
            attempts += 1
            # Step 1: Pehle sab deewar (1) yaani buildings hain
            maze = np.ones((size, size), dtype=int)

            # Step 2: Seedhi Sarakein (Horizontal Roads) kaatna
            for r in range(0, size, spacing):
                maze[r, :] = 0

            # Step 3: Seedhi Sarakein (Vertical Roads) kaatna
            for c in range(0, size, spacing):
                maze[:, c] = 0

            # Step 4: Map ko thoda natural look dene ke liye random "Dead Ends" ya blocks banana
            # (Taake har sarak bilkul boring aur seedhi na ho)
            for r in range(size):
                for c in range(size):
                    if maze[r, c] == 0 and (r % spacing != 0 and c % spacing != 0):
                        pass # Chowk ko nahi chidna
                    elif maze[r, c] == 0 and random.random() < 0.15:
                        # 15% chance hai ki sarak ke darmiyan koi block/deewar aa jaye
                        # Lekin dhyan rahe ke Start/Goal block na ho
                        maze[r, c] = 1

            # Step 5: Roads (0) par se random Start aur Goal chunna
            road_coords = [(r, c) for r in range(size) for c in range(size) if maze[r, c] == 0]

            if len(road_coords) < 2:
                continue

            start, goal = random.sample(road_coords, 2)

            # Distance check (Start aur Goal door hone chahiye)
            manhattan_dist = abs(start[0] - goal[0]) + abs(start[1] - goal[1])
            if manhattan_dist >= (size // 1.5):
                paths = cls.count_distinct_paths(maze, start, goal, max_paths_to_find=5)
                if paths >= 5:
                    print(f"City Map generated successfully in {attempts} attempts!")
                    return maze, start, goal

# --- Run aur Print karne ka code ---
if __name__ == "__main__":
    size = Config.MAZE_SIZE
    spacing = Config.STREET_SPACING

    maze, start, goal = RoadMazeGenerator.generate_road_network(size, spacing)

    print("\n🗺️ YOUR ROAD MAP ( █ = Buildings/Blocks,  . = Roads/Streets )")
    print(f"🛫 Start: {start} | 🛬 Goal: {goal}\n")

    for r in range(size):
        row_str = ""
        for c in range(size):
            if (r, c) == start:
                row_str += " 🛫 "  # Start
            elif (r, c) == goal:
                row_str += " 🛬 "  # Goal
            elif maze[r, c] == 1:
                row_str += " ██ "  # Building Block
            else:
                row_str += " .. "  # Clear Road
        print(row_str)

City Map generated successfully in 5 attempts!

🗺️ YOUR ROAD MAP ( █ = Buildings/Blocks,  . = Roads/Streets )
🛫 Start: (3, 0) | 🛬 Goal: (9, 8)

 ..  ..  ..  ..  ..  ..  ..  ..  ██  ..  ..  ..  ..  ..  .. 
 ██  ██  ██  ..  ██  ██  ..  ██  ██  ..  ██  ██  ..  ██  ██ 
 ..  ██  ██  ..  ██  ██  ..  ██  ██  ..  ██  ██  ..  ██  ██ 
 🛫  ..  ..  ..  ..  ██  ..  ██  ..  ..  ..  ..  ..  ..  .. 
 ..  ██  ██  ..  ██  ██  ..  ██  ██  ..  ██  ██  ..  ██  ██ 
 ..  ██  ██  ██  ██  ██  ..  ██  ██  ██  ██  ██  ██  ██  ██ 
 ..  ..  ..  ..  ██  ..  ..  ..  ..  ██  ██  ..  ..  ..  .. 
 ..  ██  ██  ..  ██  ██  ██  ██  ██  ..  ██  ██  ..  ██  ██ 
 ██  ██  ██  ..  ██  ██  ..  ██  ██  ..  ██  ██  ██  ██  ██ 
 ..  ..  ..  ..  ..  ..  ..  ..  🛬  ..  ..  ..  ..  ..  .. 
 ..  ██  ██  ..  ██  ██  ..  ██  ██  ..  ██  ██  ..  ██  ██ 
 ..  ██  ██  ..  ██  ██  ..  ██  ██  ..  ██  ██  ..  ██  ██ 
 ..  ..  ..  ..  ..  ..  ..  ..  ..  ..  ██  ..  ..  ..  ██ 
 ..  ██  ██  ..  ██  ██  ██  ██  ██  ..  ██  ██  ..  ██  ██ 
 .. 